In [ ]:
#!pip install gptqmodel

In [ ]:
from collections import defaultdict, deque
from sklearn.decomposition import PCA
import heapq
import math
import numpy as np
from sklearn.cluster import KMeans
import networkx as nx
import torch
import pickle
import scipy.sparse as sp
from pathlib import Path
import h5py

def build_global_clusters(all_embeddings: np.ndarray, n_clusters: int = 200):
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(all_embeddings)
    return kmeans

def build_attention_graph(
    trace_df: str,
    model,
    tokenizer,
    out_path,
    layer: int = -1,
    head_aggregation: str = 'mean',
    threshold: float = 0.2,
    batch_size: int = 1,
    hidden_layer: int = -1,
    pca_dim = 768,
    k=2,
    top_tokens_count=20):


    model.eval()
    texts = trace_df['trace'].tolist()
    ids = trace_df['id'].tolist()

    metadata = []

    h5_path = Path('graph_dir') / 'attention_graphs_with_hidden_5120_topk2_20_nodes_math500_DeepSeek32B.h5'
    ensure_dir(h5_path)

    with h5py.File(h5_path, 'w') as h5f:
      for i in tqdm(range(0, len(texts), batch_size), 'Processing generated texts...'):
        batch_texts = texts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]

        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, max_length=3000, padding=True)

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_attentions=True, output_hidden_states=True)

        layer_att = outputs.attentions[layer]
        if head_aggregation == 'mean':
            att_matrix = layer_att.mean(dim=1).to(torch.float32).cpu().numpy().astype(np.float32)
        elif head_aggregation == 'max':
            att_matrix = layer_att.max(dim=1).to(torch.float32).cpu().numpy().astype(np.float32)
        else:
            att_matrix = layer_att.mean(dim=1).to(torch.float32).cpu().numpy().astype(np.float32)

        del layer_att

        hidden_states = outputs.hidden_states[hidden_layer].to(torch.float32).cpu().numpy().astype(np.float32)

        for j, (id, text, att_mat, hs) in enumerate(zip(batch_ids, batch_texts, att_matrix, hidden_states)):
              tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][j])

              G = create_graph_from_attention(
                  tokens, att_mat, hs,
                  graph_id=id,
                  h5_file = h5f,
                  tokenizer = tokenizer,
                  k=k
              )

              metadata.append({
                    'id': id,
                    'h5_group': f'graph_{id}',
                    'num_nodes': G['num_nodes'],
                    'num_edges': G['num_edges'],
                    'k': k,
                    'top_tokens_count': top_tokens_count,
                })

        del outputs, att_matrix, hidden_states, inputs
        torch.cuda.empty_cache()


    with open('metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    return metadata



def create_graph_from_attention(tokens, att_matrix, hidden_states, graph_id, h5_file, tokenizer, threshold=0.2, k=2,
                                pca_dim=500, top_tokens_count=20):

    skip_tokens = {'▁', '</s>', '<s>', '<unk>', '.', ',', '!', '?', ';', ':'}

    valid_mask = np.array([
        token not in skip_tokens and not (token.startswith('<') and token.endswith('>'))
        for token in tokens
    ])

    valid_indices = np.where(valid_mask)[0]

    if len(valid_indices) < 2:
        return {
            'graph_file': None,
            'nodes_file': None,
            'num_nodes': 0,
            'num_edges': 0}

    valid_att_submatrix = att_matrix[np.ix_(valid_indices, valid_indices)].astype(np.float32)

    received_attention_matrix  = np.sum(valid_att_submatrix, axis = 0)
    local_valid_indices = np.array(sorted(np.argsort(received_attention_matrix)[-top_tokens_count:]))
    global_valid_indices = valid_indices[local_valid_indices]

    att_submatrix = att_matrix[np.ix_(local_valid_indices, local_valid_indices)].astype(np.float32)

    seq_len = len(local_valid_indices)

    k_eff = min(k + 1, seq_len)
    topk_indices = np.argpartition(-att_submatrix, k_eff, axis=1)[:, :k_eff]
    topk_values = np.take_along_axis(att_submatrix, topk_indices, axis=1)

    diag_mask = (topk_indices != np.arange(seq_len)[:, None])

    rows = np.repeat(np.arange(seq_len), k_eff)[diag_mask.ravel()]
    cols = topk_indices.ravel()[diag_mask.ravel()]
    vals = topk_values.ravel()[diag_mask.ravel()]

    pos_mask = vals > 0
    rows, cols, vals = rows[pos_mask], cols[pos_mask], vals[pos_mask]\

    if len(rows) == 0:
        csr = sp.csr_matrix((seq_len, seq_len))
    else:
        csr = sp.csr_matrix((vals, (rows, cols)), shape=(seq_len, seq_len))

    hs_top = hidden_states[global_valid_indices]

    group_name = f'graph_{graph_id}'

    grp = h5_file.create_group(group_name)

    if len(rows) == 0:
        csr = sp.csr_matrix((seq_len, seq_len))
    else:
        csr = sp.csr_matrix((vals, (rows, cols)), shape=(seq_len, seq_len))

    grp.create_dataset('adj_data', data=csr.data, compression='gzip', compression_opts=9)
    grp.create_dataset('adj_indices', data=csr.indices, compression='gzip', compression_opts=9)
    grp.create_dataset('adj_indptr', data=csr.indptr, compression='gzip', compression_opts=9)
    grp.create_dataset('adj_shape', data=csr.shape)

    grp.create_dataset('valid_indices', data=valid_indices.astype(np.int32), compression='gzip', compression_opts=9)
    grp.create_dataset('local_valid_indices', data=local_valid_indices.astype(np.int32), compression='gzip', compression_opts=9)

    token_ids = [tokenizer.convert_tokens_to_ids(tokens[i]) for i in global_valid_indices]
    grp.create_dataset('token_ids', data=np.array(token_ids, dtype=np.int32), compression='gzip', compression_opts=9)
    grp.create_dataset('hidden_states', data=hs_top,
                      compression='gzip', compression_opts=9)

    grp.attrs['num_nodes'] = seq_len
    grp.attrs['num_edges'] = len(rows)
    grp.attrs['graph_id'] = graph_id
    grp.attrs['top_tokens_count'] = seq_len
    grp.attrs['k_neighbors'] = k
    print(seq_len)


    return {
        'num_nodes': seq_len,
        'num_edges': len(rows),
        'top_tokens_count': seq_len,
    }



def analyze_graph_simple(G):

    G_und = G.to_undirected()

    n_nodes = G_und.number_of_nodes()
    n_edges = G_und.number_of_edges()

    degree_centrality = nx.degree_centrality(G_und)

    node_size = [
      v * 1000 for v in degree_centrality.values()
    ]

    metrics = {}

    metrics['n_nodes'] = n_nodes
    metrics['n_edges'] = n_edges
    metrics['density'] = nx.density(G_und) if n_nodes > 0 else 0
    metrics['edge_to_node_ratio'] = n_edges / n_nodes if n_nodes > 0 else 0

    metrics['is_connected'] = int(nx.is_connected(G_und) if n_nodes > 0 else False)
    metrics['is_strongly_connected'] = int(nx.is_strongly_connected(G) if n_nodes > 0 else False)
    metrics['n_connected_components'] = nx.number_connected_components(G_und) if n_nodes > 0 else 0
    metrics['n_strongly_connected'] = nx.number_strongly_connected_components(G) if n_nodes > 0 else 0

    if n_nodes > 0:
        component_sizes = [len(c) for c in nx.connected_components(G_und)]
        metrics['largest_component_size'] = max(component_sizes) if component_sizes else 0
    else:
        metrics['largest_component_size'] = 0

    if n_nodes > 0:
        degree_seq = [d for n, d in G_und.degree()]
        in_degree_seq = [d for n, d in G.in_degree()]
        out_degree_seq = [d for n, d in G.out_degree()]

        metrics['avg_degree'] = np.mean(degree_seq)
        metrics['median_degree'] = np.median(degree_seq)
        metrics['std_degree'] = np.std(degree_seq)
        metrics['min_degree'] = min(degree_seq)
        metrics['max_degree'] = max(degree_seq)
        metrics['degree_variance'] = np.var(degree_seq)
        metrics['avg_in_degree'] = np.mean(in_degree_seq) if in_degree_seq else 0
        metrics['avg_out_degree'] = np.mean(out_degree_seq) if out_degree_seq else 0
        metrics['degree_assortativity'] = nx.degree_assortativity_coefficient(G_und)
        probs = np.bincount(degree_seq) / n_nodes
        probs = probs[probs > 0]
        metrics['degree_entropy'] = -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0
    else:
        metrics.update({
            'avg_degree': 0, 'median_degree': 0, 'std_degree': 0, 'min_degree': 0,
            'max_degree': 0, 'degree_variance': 0, 'avg_in_degree': 0, 'avg_out_degree': 0,
            'degree_assortativity': 0, 'degree_entropy': 0
        })

    if n_nodes > 0:
        metrics['avg_clustering'] = nx.average_clustering(G_und)
        metrics['transitivity'] = nx.transitivity(G_und)
        metrics['n_triangles'] = sum(nx.triangles(G_und).values()) // 3
    else:
        metrics['avg_clustering'] = 0
        metrics['transitivity'] = 0
        metrics['n_triangles'] = 0

    if nx.is_connected(G_und) and n_nodes > 1:
        metrics['diameter'] = nx.diameter(G_und)
        metrics['radius'] = nx.radius(G_und)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_und)

        ecc = nx.eccentricity(G_und)
        metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
        metrics['max_eccentricity'] = max(ecc.values())
        metrics['min_eccentricity'] = min(ecc.values())
    else:
        if n_nodes > 0:
            largest_cc = max(nx.connected_components(G_und), key=len)
            subgraph = G_und.subgraph(largest_cc)
            if len(subgraph) > 1:
                metrics['diameter'] = nx.diameter(subgraph)
                metrics['radius'] = nx.radius(subgraph)
                metrics['avg_path_length'] = nx.average_shortest_path_length(subgraph)
                ecc = nx.eccentricity(subgraph)
                metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
                metrics['max_eccentricity'] = max(ecc.values())
                metrics['min_eccentricity'] = min(ecc.values())
            else:
                metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                               'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})
        else:
            metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                           'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})

    try:
        if n_nodes > 0:
            has_loop = not nx.is_forest(G_und) if n_nodes > 0 else False
            if has_loop:
                n_components = nx.number_connected_components(G_und)
                loop_count = n_edges - n_nodes + n_components
            else:
                loop_count = 0
        else:
            has_loop = False
            loop_count = 0
    except:
        has_loop = False
        loop_count = 0

    metrics['has_loop'] = has_loop
    metrics['loop_count'] = loop_count

    if n_nodes > 1 and metrics['avg_clustering'] > 0 and metrics['avg_path_length'] > 0:
        N = n_nodes
        K = metrics['avg_degree']

        if K > 1 and N > 1:
            C_rand = K / (N - 1) if N > 1 else 0
            L_rand = np.log(N) / np.log(K) if K > 1 else float('inf')
            clustering_norm = metrics['avg_clustering'] / C_rand if C_rand > 0 else 0
            path_length_norm = metrics['avg_path_length'] / L_rand if L_rand not in [0, float('inf')] else 0
            small_world_index = clustering_norm / path_length_norm if path_length_norm > 0 else 0
        else:
            small_world_index = 0
    else:
        small_world_index = 0

    metrics['small_world_index'] = small_world_index

    if n_nodes > 0:
        deg_cent = nx.degree_centrality(G_und)
        metrics['avg_degree_centrality'] = np.mean(list(deg_cent.values()))
        metrics['max_degree_centrality'] = max(deg_cent.values())
        metrics['std_degree_centrality'] = np.std(list(deg_cent.values()))

        in_deg_cent = nx.in_degree_centrality(G)
        out_deg_cent = nx.out_degree_centrality(G)
        metrics['avg_in_degree_centrality'] = np.mean(list(in_deg_cent.values())) if in_deg_cent else 0
        metrics['avg_out_degree_centrality'] = np.mean(list(out_deg_cent.values())) if out_deg_cent else 0

        bet_cent = nx.betweenness_centrality(G_und)
        metrics['avg_betweenness'] = np.mean(list(bet_cent.values()))
        metrics['max_betweenness'] = max(bet_cent.values())
        metrics['std_betweenness'] = np.std(list(bet_cent.values()))

        if any('weight' in d for (_, _, d) in G.edges(data=True)):
            bet_cent_weighted = nx.betweenness_centrality(G_und, weight='weight')
            metrics['avg_betweenness_weighted'] = np.mean(list(bet_cent_weighted.values()))
            metrics['max_betweenness_weighted'] = max(bet_cent_weighted.values())
        else:
            metrics['avg_betweenness_weighted'] = metrics['avg_betweenness']
            metrics['max_betweenness_weighted'] = metrics['max_betweenness']

        if nx.is_connected(G_und):
            clo_cent = nx.closeness_centrality(G_und)
        else:
            clo_cent = nx.harmonic_centrality(G_und)
        metrics['avg_closeness'] = np.mean(list(clo_cent.values()))
        metrics['max_closeness'] = max(clo_cent.values())

        try:
            eig_cent = nx.eigenvector_centrality(G_und, max_iter=1000)
            metrics['avg_eigenvector'] = np.mean(list(eig_cent.values()))
            metrics['max_eigenvector'] = max(eig_cent.values())
        except:
            metrics['avg_eigenvector'] = 0
            metrics['max_eigenvector'] = 0

        try:
            katz_cent = nx.katz_centrality(G_und, alpha=0.1)
            metrics['avg_katz'] = np.mean(list(katz_cent.values()))
            metrics['max_katz'] = max(katz_cent.values())
        except:
            metrics['avg_katz'] = 0
            metrics['max_katz'] = 0

    else:
        null_cent = ['avg_degree_centrality', 'max_degree_centrality', 'std_degree_centrality',
                    'avg_in_degree_centrality', 'avg_out_degree_centrality', 'avg_betweenness',
                    'max_betweenness', 'std_betweenness', 'avg_betweenness_weighted', 'max_betweenness_weighted',
                    'avg_closeness', 'max_closeness', 'avg_eigenvector', 'max_eigenvector',
                    'avg_katz', 'max_katz']
        for c in null_cent:
            metrics[c] = 0
    return G, metrics

In [ ]:
import argparse
import json
from pathlib import Path
from typing import Iterable
import sys
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import torch

import numpy as np
from tqdm import tqdm


def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def load_npz(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(path) as data:
        return {k: data[k] for k in data.files}


def process_generated_reasonings(
    traces_path: Path,
    out_path: Path,
    graph_dir: Path,
    model,
    tokenizer
) -> int:

    ensure_dir(out_path)

    traces = []
    count = 0
    with open(traces_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                trace =  json.loads(line)
                traces.append(trace)

    traces_df = pd.DataFrame(traces)
    traces_df = traces_df.dropna(subset=['trace'])

    traces_df = traces_df[traces_df['trace'] != '']

    metadata = build_attention_graph(traces_df, model, tokenizer, out_path)
    return metadata



def setup_model_for_attention(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        output_attentions=True,
        use_cache=False
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def main() -> None:
    model_name ="Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ"
    model, tokenizer = setup_model_for_attention(model_name)
    split = "trace"
    traces_path = "/content/all_traces_math_qwen32b_vllm_final.jsonl"
    features_root = "data/math_graph"
    graph_dir = Path("graph_dir")
    out = None


    model_slug = model_name
    split = split

    traces_path = Path(traces_path)

    features_out = Path(out) if out else Path(features_root) / split / f"attention_graph_{model_slug}_math500_all_graph_features.jsonl"

    metadata = process_generated_reasonings(traces_path, features_out, graph_dir, model, tokenizer)
    print(f"Wrote {split} graph features for {len(metadata)} items to {features_out}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] `torch.bfloat16` is not supported for AWQ CUDA/XPU kernels yet. Casting to `torch.float16`.
[transformers] The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


...

20


Processing generated texts...: 100%|█████████▉| 499/500 [07:18<00:01,  1.53s/it]

20


Processing generated texts...: 100%|██████████| 500/500 [07:19<00:00,  1.14it/s]

20
Wrote trace graph features for 500 items to data/math_graph/trace/attention_graph_Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ_math500_all_graph_features.jsonl


In [ ]:
from google.colab import files

files.download('/content/graph_dir/attention_graphs_with_hidden_5120_topk2_20_nodes_math500_DeepSeek32B.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>